# DistilBERT on WikiText — encoder activations in memory

Every transformer block outputs `(batch, seq_len, hidden)`, so a text run costs
far more per sample than a vision one. Downloads on first run: WikiText-2
(~5 MB) and DistilBERT weights (~250 MB).

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

from nnact import ActivationMapper
from nnact.utils import WikiTextSamples, activation_loader

MODEL = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModel.from_pretrained(MODEL)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
dataset = WikiTextSamples(tokenizer, n=256, max_length=64)
print(f"{len(dataset)} passages | input_ids {tuple(dataset[0]['input_ids'].shape)}")
print(f"real tokens in first: {int(dataset.encoded['attention_mask'][0].sum())}")
print(dataset.texts[0][:90], "...")


256 passages | input_ids (64,)
real tokens in first: 64
Robert Boulter is an English film , television and theatre actor . He had a guest @-@ star ...


In [3]:
mapper = ActivationMapper(model)

# depth=3 reaches the individual blocks; depth=1 would only show
# "embeddings" and "transformer".
mapper.summary(depth=3).head(12)

,module,parameters
layer,,
embeddings,Embeddings,23835648
embeddings.word_embeddings,Embedding,23440896
embeddings.position_embeddings,Embedding,393216
embeddings.LayerNorm,LayerNorm,1536
embeddings.dropout,Dropout,0
transformer,Transformer,42527232
transformer.layer,ModuleList,42527232
transformer.layer.0,TransformerBlock,7087872
transformer.layer.1,TransformerBlock,7087872


In [4]:
# Embeddings, an early block, a middle block, and the last one.
loader = activation_loader(dataset, batch_size=32)
store = mapper.map(
    loader,
    ["embeddings", "transformer.layer.0", "transformer.layer.3", "transformer.layer.5"],
)
store.metadata


DistilBertModel activations[1/8]  12%|#2         [00:00<?]

RunMetadata(
    model      = DistilBertModel
    parameters = 66.4M
    layers     = embeddings, transformer.layer.0, transformer.layer.3, transformer.layer.5
    samples    = 256
    batch_size = 32
    device     = cpu
    seconds    = 1.67s
    throughput = 152/s
    created    = 2026-09-14T19:21:00+00:00
)

In [5]:
# (n_samples, seq_len, hidden) per layer - seq_len is why text is expensive.
store.summary()

,shape,elements,bytes
layer,,,
embeddings,"(64, 768)",49152,50331648
transformer.layer.0,"(64, 768)",49152,50331648
transformer.layer.3,"(64, 768)",49152,50331648
transformer.layer.5,"(64, 768)",49152,50331648


In [6]:
# Row i of a layer is one passage's whole token sequence.
last = store.activations["transformer.layer.5"]
print("layer 5 stacked:", tuple(last.shape))

# Most uses want one vector per passage. Mean-pool over real tokens only,
# so padding does not drag the average toward zero.
mask = dataset.encoded["attention_mask"].unsqueeze(-1).float()
pooled = (last * mask).sum(1) / mask.sum(1)
print("mean-pooled:", tuple(pooled.shape), "| CLS:", tuple(last[:, 0].shape))

layer 5 stacked: (256, 64, 768)
mean-pooled: (256, 768) | CLS: (256, 768)


In [7]:
# How far each layer's representation sits from the final one.
for name in store.layer_names:
    vec = (store.activations[name] * mask).sum(1) / mask.sum(1)
    sim = torch.nn.functional.cosine_similarity(vec, pooled, dim=1).mean()
    print(f"  {name:<22} cos(layer, final) = {sim:.3f}")

  embeddings             cos(layer, final) = 0.357
  transformer.layer.0    cos(layer, final) = 0.470
  transformer.layer.3    cos(layer, final) = 0.613
  transformer.layer.5    cos(layer, final) = 1.000
